In [1]:
import glob
from os.path import getctime, join
from utils.files import load_json
import pandas as pd

In [2]:

repeat_experiments = [
    'logs/hyper-param-tune-2025-10-05 14:10:07.433826',
    'logs/hyper-param-tune-2025-10-07 11:53:41.690422',
    'logs/hyper-param-tune-2025-11-12 16:12:13.321121',
    'logs/hyper-param-tune-2025-11-11 17:36:08.759505'
]

#dataset_fqe_value = -9.43
def load_latest_results(exp_path, results_type):
    result_files = glob.glob(join(exp_path, 'eval', results_type, '**', 'results.json'), recursive=True)
    result_files.sort(key=getctime)
    return load_json(result_files[0])

def flatten(lol):
   for item in lol:
       yield from item
       
def load_experiments_paths(root_path):
    exps = glob.glob(join(root_path, '*', '*'), recursive=True)
    
    filtered_exps = []
    for exp in exps:
        if load_json(join(exp, 'experiment_config.json'))['finished']:
            filtered_exps.append(exp)
    return filtered_exps

experiments = list(flatten([load_experiments_paths(root_path) for root_path in repeat_experiments]))

print(len(experiments))

586


In [3]:
dfs = []
for experiment in experiments:
    ae_results = load_latest_results(experiment, results_type='Hybrid-Autencoder')
    fqe_results = load_latest_results(experiment, results_type='Dist-fqe')
    config = load_json(join(experiment, 'config.json'))
    algo_name = config['name']#f"{config['name']}(alpha={config['alpha']})" if config['name'] == 'CQL' else config['name']


    experiment_data = [{
        "exp_id": experiment,
        "checkpoint": int(key),
        "algo": algo_name,
        "ood_loss": -val['loss'],
        "fqe_value": fqe_results[key]['value_mean'],
        "root_exp": experiment.split('/')[-3],
    } for key, val in ae_results.items()]
    
    df = pd.DataFrame(experiment_data)
    dfs.append(df)
    

df = pd.concat(dfs)

Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL
Hybrid-IQL

In [4]:
df

,exp_id,checkpoint,algo,ood_loss,fqe_value,root_exp
0,logs/hyper-param-tune-2025-10-05 14:10:07.4338...,100000,Hybrid-IQL,-0.325668,-6.515283,hyper-param-tune-2025-10-05 14:10:07.433826
0,logs/hyper-param-tune-2025-10-05 14:10:07.4338...,100000,Hybrid-IQL,-0.388485,-6.717375,hyper-param-tune-2025-10-05 14:10:07.433826
0,logs/hyper-param-tune-2025-10-05 14:10:07.4338...,100000,Hybrid-IQL,-0.377777,-5.448385,hyper-param-tune-2025-10-05 14:10:07.433826
0,logs/hyper-param-tune-2025-10-05 14:10:07.4338...,100000,Hybrid-IQL,-0.432791,-3.996110,hyper-param-tune-2025-10-05 14:10:07.433826
0,logs/hyper-param-tune-2025-10-05 14:10:07.4338...,100000,Hybrid-IQL,-0.415276,-8.364567,hyper-param-tune-2025-10-05 14:10:07.433826
...,...,...,...,...,...,...
0,logs/hyper-param-tune-2025-11-11 17:36:08.7595...,100000,Discrete-IQL,-0.414091,-7.386868,hyper-param-tune-2025-11-11 17:36:08.759505
0,logs/hyper-param-tune-2025-11-11 17:36:08.7595...,100000,Discrete-IQL,-0.463168,-9.731789,hyper-param-tune-2025-11-11 17:36:08.759505
0,logs/hyper-param-tune-2025-11-11 17:36:08.7595...,100000,Discrete-IQL,-0.400002,-6.957433,hyper-param-tune-2025-11-11 17:36:08.759505
0,logs/hyper-param-tune-2025-11-11 17:36:08.7595...,100000,Discrete-IQL,-0.381413,-6.851023,hyper-param-tune-2025-11-11 17:36:08.759505


In [5]:
checkpoint_map = {
    'CQL': 100000,
    'C-CQL': 100000,
    'CF-CQL': 100000,
    'Hybrid-EDAC': 100000,
    'Hybrid-IQL': 100000
}

In [6]:

def get_checkpoint_stats(df):
    df = df.copy(deep=True)
    mask = df.apply(
        lambda row: row['checkpoint'] == checkpoint_map.get(row['algo'], 100000),
        axis=1
    )
    filtered = df[mask]
    return filtered

df = df[(df['fqe_value'] <= 0) & (df['fqe_value'] >= -15)]


filtered_df = get_checkpoint_stats(df)
filtered_df

,exp_id,checkpoint,algo,ood_loss,fqe_value,root_exp
0,logs/hyper-param-tune-2025-10-05 14:10:07.4338...,100000,Hybrid-IQL,-0.325668,-6.515283,hyper-param-tune-2025-10-05 14:10:07.433826
0,logs/hyper-param-tune-2025-10-05 14:10:07.4338...,100000,Hybrid-IQL,-0.388485,-6.717375,hyper-param-tune-2025-10-05 14:10:07.433826
0,logs/hyper-param-tune-2025-10-05 14:10:07.4338...,100000,Hybrid-IQL,-0.377777,-5.448385,hyper-param-tune-2025-10-05 14:10:07.433826
0,logs/hyper-param-tune-2025-10-05 14:10:07.4338...,100000,Hybrid-IQL,-0.432791,-3.996110,hyper-param-tune-2025-10-05 14:10:07.433826
0,logs/hyper-param-tune-2025-10-05 14:10:07.4338...,100000,Hybrid-IQL,-0.415276,-8.364567,hyper-param-tune-2025-10-05 14:10:07.433826
...,...,...,...,...,...,...
0,logs/hyper-param-tune-2025-11-11 17:36:08.7595...,100000,Discrete-IQL,-0.414091,-7.386868,hyper-param-tune-2025-11-11 17:36:08.759505
0,logs/hyper-param-tune-2025-11-11 17:36:08.7595...,100000,Discrete-IQL,-0.463168,-9.731789,hyper-param-tune-2025-11-11 17:36:08.759505
0,logs/hyper-param-tune-2025-11-11 17:36:08.7595...,100000,Discrete-IQL,-0.400002,-6.957433,hyper-param-tune-2025-11-11 17:36:08.759505
0,logs/hyper-param-tune-2025-11-11 17:36:08.7595...,100000,Discrete-IQL,-0.381413,-6.851023,hyper-param-tune-2025-11-11 17:36:08.759505


In [7]:
filtered_df.groupby('algo')[['ood_loss', 'fqe_value']].agg(['mean', 'std'])

ood_loss           fqe_value          
                  mean       std      mean       std
algo                                                
C-CQL        -0.913884  0.363413 -8.105434  1.245923
CF-CQL       -0.621045  0.299768 -9.109610  1.980027
CQL          -0.907928  0.368842 -8.245743  1.203979
Discrete-IQL -0.413102  0.027155 -7.231695  0.858021
Hybrid-EDAC  -1.607126  0.541118 -7.135462  3.631157
Hybrid-IQL   -0.380360  0.039509 -6.186393  1.231748